# Deliverable 3 training and evaluation
Extracted Deliverable 3 cells with their original executed outputs. The original combined upload is retained separately. Use this notebook for D3 reproduction; do not run the historical D2 section in the original upload. Select a fresh output directory in the first code cell when reproducing. Existing seed directories intentionally stop a rerun. Install requirements-train.txt in a GPU Colab runtime first and preserve the original seed-0 results.


In [1]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import platform
import torch

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Colab Notebooks/AML_GNN_Project"
)
DATA_DIR = PROJECT_DIR / "data"
D2_RESULTS = PROJECT_DIR / "results"
D3_RESULTS = PROJECT_DIR / "results_deliverable3"
D3_RESULTS.mkdir(parents=True, exist_ok=True)

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("\nOriginal dataset:")
for filename in [
    "elliptic_txs_features.csv",
    "elliptic_txs_classes.csv",
    "elliptic_txs_edgelist.csv",
]:
    path = DATA_DIR / filename
    print(filename, "FOUND" if path.is_file() else "MISSING")

print("\nDeliverable 2 baseline:")
for prefix in ["xgb", "mlp", "graphsage"]:
    filename = f"{prefix}_adaptation_seed0_metrics.csv"
    path = D2_RESULTS / filename
    print(filename, "FOUND" if path.is_file() else "MISSING")

print("\nNew results directory:", D3_RESULTS)

Mounted at /content/drive
Python: 3.13.16
PyTorch: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4

Original dataset:
elliptic_txs_features.csv FOUND
elliptic_txs_classes.csv FOUND
elliptic_txs_edgelist.csv FOUND

Deliverable 2 baseline:
xgb_adaptation_seed0_metrics.csv FOUND
mlp_adaptation_seed0_metrics.csv FOUND
graphsage_adaptation_seed0_metrics.csv FOUND

New results directory: /content/drive/MyDrive/Colab Notebooks/AML_GNN_Project/results_deliverable3


In [2]:
import json
import numpy as np
import pandas as pd
from IPython.display import display

# Load the original data.
features = pd.read_csv(
    DATA_DIR / "elliptic_txs_features.csv", header=None
)
classes = pd.read_csv(
    DATA_DIR / "elliptic_txs_classes.csv",
    dtype={"class": str}
)
edges = pd.read_csv(DATA_DIR / "elliptic_txs_edgelist.csv")

local_cols = [f"local_{i:02d}" for i in range(93)]
aggregate_cols = [f"aggregate_{i:02d}" for i in range(72)]

assert features.shape[1] == 167
features.columns = (
    ["txId", "time_step"] + local_cols + aggregate_cols
)

# Validate identifiers and labels before merging.
assert not features["txId"].duplicated().any()
assert not classes["txId"].duplicated().any()
assert set(features["txId"]) == set(classes["txId"])
assert set(classes["class"]) == {"1", "2", "unknown"}

nodes = features.merge(
    classes, on="txId", validate="one_to_one", sort=False
)

# Unknown labels remain excluded from supervised targets.
nodes["y"] = nodes["class"].map(
    {"1": 1, "2": 0, "unknown": -1}
).astype(int)

assert np.isfinite(
    nodes[local_cols + aggregate_cols].to_numpy(dtype=float)
).all()

assert (
    nodes["time_step"].between(1, 49)
    & nodes["time_step"].mod(1).eq(0)
).all()

# Validate graph endpoints and snapshot boundaries.
assert list(edges.columns) == ["txId1", "txId2"]
assert not edges.duplicated().any()

node_ids = set(nodes["txId"])
assert set(edges["txId1"]).issubset(node_ids)
assert set(edges["txId2"]).issubset(node_ids)

time_lookup = nodes.set_index("txId")["time_step"]
cross_time_edges = int((
    edges["txId1"].map(time_lookup)
    != edges["txId2"].map(time_lookup)
).sum())
assert cross_time_edges == 0

# Preserve the Deliverable 2 split.
split_rows = []
for name, first, last in [
    ("Training", 1, 30),
    ("Validation", 31, 34),
    ("Testing", 35, 49),
]:
    subset = nodes[nodes["time_step"].between(first, last)]
    split_rows.append({
        "partition": name,
        "steps": f"{first}-{last}",
        "total_nodes": len(subset),
        "labeled": int(subset["y"].ge(0).sum()),
        "illicit": int(subset["y"].eq(1).sum()),
        "licit": int(subset["y"].eq(0).sum()),
        "unknown": int(subset["y"].eq(-1).sum()),
    })

split_summary = pd.DataFrame(split_rows)

# Fail early if the data differ from the verified baseline.
assert len(nodes) == 203769
assert len(edges) == 234355
assert split_summary["labeled"].tolist() == [26905, 2989, 16670]
assert split_summary["illicit"].tolist() == [2954, 508, 1083]

counts_by_step = pd.crosstab(
    nodes["time_step"], nodes["class"]
).reindex(index=range(1, 50), columns=["1", "2", "unknown"],
          fill_value=0)
counts_by_step.columns = ["illicit", "licit", "unknown"]

# Write only to the new Deliverable 3 directory.
split_summary.to_csv(D3_RESULTS / "split_summary.csv", index=False)
counts_by_step.to_csv(D3_RESULTS / "counts_by_step.csv")

schema = {
    "predictor_columns": local_cols,
    "excluded_columns": (
        ["txId", "time_step", "class", "y"] + aggregate_cols
    ),
    "label_mapping": {"1": 1, "2": 0, "unknown": -1},
    "split": {"train": [1, 30], "validation": [31, 34],
              "test": [35, 49]},
    "cross_time_edges": cross_time_edges,
    "neural_scaler_rule": "Fit on labeled training nodes only",
}
(D3_RESULTS / "feature_schema.json").write_text(
    json.dumps(schema, indent=2)
)

print("Deliverable 3 data checks PASSED")
print("Transactions:", len(nodes))
print("Edges:", len(edges))
print("Predictors:", len(local_cols))
print("Cross-time edges:", cross_time_edges)
display(split_summary)
print("Saved preprocessing records to:", D3_RESULTS)

Deliverable 3 data checks PASSED
Transactions: 203769
Edges: 234355
Predictors: 93
Cross-time edges: 0


,partition,steps,total_nodes,labeled,illicit,licit,unknown
0,Training,1-30,123287,26905,2954,23951,96382
1,Validation,31-34,12978,2989,508,2481,9989
2,Testing,35-49,67504,16670,1083,15587,50834


Saved preprocessing records to: /content/drive/MyDrive/Colab Notebooks/AML_GNN_Project/results_deliverable3


XGBOOST

In [3]:
import time
import platform
import xgboost as xgb

from sklearn.metrics import average_precision_score, f1_score

assert torch.cuda.is_available(), "Select a GPU runtime first."

# Preserve the original temporal split.
train_mask = nodes["time_step"].between(1, 30) & nodes["y"].ge(0)
val_mask = nodes["time_step"].between(31, 34) & nodes["y"].ge(0)

X_train = nodes.loc[train_mask, local_cols].astype("float32")
y_train = nodes.loc[train_mask, "y"]
X_val = nodes.loc[val_mask, local_cols].astype("float32")
y_val = nodes.loc[val_mask, "y"]

assert X_train.columns.tolist() == local_cols
assert X_val.columns.tolist() == local_cols

FIXED_CLASS_WEIGHT = float(
    y_train.eq(0).sum() / y_train.eq(1).sum()
)

XGB_D3_DIR = D3_RESULTS / "xgboost"
XGB_D3_DIR.mkdir(exist_ok=True)

# Protect completed runs against accidental replacement.
for seed in [1, 2]:
    run_dir = XGB_D3_DIR / f"seed_{seed}"
    assert not run_dir.exists(), (
        f"{run_dir} already exists. Preserve it before rerunning."
    )

validation_rows = []

for seed in [1, 2]:
    run_dir = XGB_D3_DIR / f"seed_{seed}"
    run_dir.mkdir()

    model = xgb.XGBClassifier(
        n_estimators=150,
        max_depth=4,
        learning_rate=0.05,
        tree_method="hist",
        device="cuda",
        objective="binary:logistic",
        eval_metric="logloss",
        scale_pos_weight=FIXED_CLASS_WEIGHT,
        random_state=seed,
        n_jobs=2,
    )

    torch.cuda.synchronize()
    start = time.perf_counter()

    model.fit(X_train, y_train)

    torch.cuda.synchronize()
    fit_seconds = time.perf_counter() - start

    val_scores = model.get_booster().predict(
        xgb.DMatrix(X_val)
    )

    assert np.isfinite(val_scores).all()

    # Select the threshold using validation labels only.
    # Prefer the larger threshold when F1 ties.
    threshold_grid = np.round(np.arange(0.05, 1.00, 0.05), 2)

    threshold_results = pd.DataFrame([
        {
            "threshold": float(threshold),
            "validation_f1": f1_score(
                y_val,
                val_scores >= threshold,
                zero_division=0,
            ),
        }
        for threshold in threshold_grid
    ])

    best = threshold_results.sort_values(
        ["validation_f1", "threshold"],
        ascending=[False, False],
    ).iloc[0]

    chosen_threshold = float(best["threshold"])
    validation_ap = float(
        average_precision_score(y_val, val_scores)
    )

    # Save model and validation evidence.
    model.save_model(str(run_dir / "frozen_model.json"))

    validation_predictions = nodes.loc[
        val_mask, ["txId", "time_step", "y"]
    ].copy()

    validation_predictions["score"] = val_scores
    validation_predictions["prediction"] = (
        val_scores >= chosen_threshold
    ).astype(int)

    validation_predictions.to_csv(
        run_dir / "validation_predictions.csv", index=False
    )
    threshold_results.to_csv(
        run_dir / "validation_threshold_search.csv", index=False
    )

    settings = {
        "model": "XGBoost",
        "seed": seed,
        "training_steps": [1, 30],
        "validation_steps": [31, 34],
        "predictor_columns": local_cols,
        "threshold": chosen_threshold,
        "validation_AP": validation_ap,
        "validation_F1": float(best["validation_f1"]),
        "positive_class_weight": FIXED_CLASS_WEIGHT,
        "fit_seconds": fit_seconds,
        "python": platform.python_version(),
        "xgboost": xgb.__version__,
        "gpu": torch.cuda.get_device_name(0),
        "parameters": {
            "n_estimators": 150,
            "max_depth": 4,
            "learning_rate": 0.05,
            "subsample": 1.0,
            "colsample_bytree": 1.0,
        },
    }

    (run_dir / "run_info.json").write_text(
        json.dumps(settings, indent=2)
    )

    validation_rows.append({
        "seed": seed,
        "validation_AP": validation_ap,
        "validation_F1": float(best["validation_f1"]),
        "threshold": chosen_threshold,
        "fit_seconds": fit_seconds,
    })

xgb_validation_summary = pd.DataFrame(validation_rows)
xgb_validation_summary.to_csv(
    XGB_D3_DIR / "validation_summary.csv", index=False
)

print("XGBoost version:", xgb.__version__)
print("Fixed positive-class weight:", FIXED_CLASS_WEIGHT)
display(xgb_validation_summary.round(4))
print("Saved to:", XGB_D3_DIR)

XGBoost version: 3.4.1
Fixed positive-class weight: 8.107989167230873


,seed,validation_AP,validation_F1,threshold,fit_seconds
0,1,0.9634,0.9264,0.7,1.4874
1,2,0.9634,0.9264,0.7,1.9430


Saved to: /content/drive/MyDrive/Colab Notebooks/AML_GNN_Project/results_deliverable3/xgboost


MLP

In [4]:
import copy
import random
import joblib
from torch import nn
from sklearn.preprocessing import StandardScaler

D3_DEVICE = torch.device("cuda")
MLP_D3_DIR = D3_RESULTS / "mlp"
MLP_D3_DIR.mkdir(exist_ok=True)

# Protect existing runs.
for seed in [1, 2]:
    assert not (MLP_D3_DIR / f"seed_{seed}").exists(), (
        f"MLP seed {seed} already exists. Preserve it before rerunning."
    )

def set_d3_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

# Fit preprocessing on labeled training nodes only.
d3_scaler = StandardScaler()
d3_scaler.fit(nodes.loc[train_mask, local_cols].astype("float32"))

assert d3_scaler.feature_names_in_.tolist() == local_cols

train_array = d3_scaler.transform(
    nodes.loc[train_mask, local_cols].astype("float32")
).astype("float32")

val_array = d3_scaler.transform(
    nodes.loc[val_mask, local_cols].astype("float32")
).astype("float32")

assert np.isfinite(train_array).all()
assert np.isfinite(val_array).all()

mlp_train_x = torch.from_numpy(train_array).to(D3_DEVICE)
mlp_val_x = torch.from_numpy(val_array).to(D3_DEVICE)

mlp_train_y = torch.tensor(
    nodes.loc[train_mask, "y"].to_numpy(),
    dtype=torch.float32,
    device=D3_DEVICE,
)
mlp_val_y = nodes.loc[val_mask, "y"].to_numpy()

joblib.dump(d3_scaler, MLP_D3_DIR / "training_scaler.joblib")

class TransactionMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(93, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        return self.network(x).squeeze(-1)

mlp_validation_rows = []

for seed in [1, 2]:
    set_d3_seed(seed)
    run_dir = MLP_D3_DIR / f"seed_{seed}"
    run_dir.mkdir()

    current_mlp = TransactionMLP().to(D3_DEVICE)
    optimizer = torch.optim.Adam(
        current_mlp.parameters(), lr=0.001, weight_decay=1e-4
    )
    criterion = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor(
            FIXED_CLASS_WEIGHT,
            dtype=torch.float32,
            device=D3_DEVICE,
        )
    )

    best_ap = -np.inf
    best_epoch = 0
    best_state = None
    no_improvement = 0
    history = []

    torch.cuda.synchronize()
    start = time.perf_counter()

    for epoch in range(1, 201):
        current_mlp.train()
        optimizer.zero_grad(set_to_none=True)

        loss = criterion(current_mlp(mlp_train_x), mlp_train_y)
        assert torch.isfinite(loss), "Non-finite training loss."
        loss.backward()
        optimizer.step()

        current_mlp.eval()
        with torch.no_grad():
            val_scores = torch.sigmoid(
                current_mlp(mlp_val_x)
            ).cpu().numpy()

        assert np.isfinite(val_scores).all()
        val_ap = float(
            average_precision_score(mlp_val_y, val_scores)
        )

        history.append({
            "epoch": epoch,
            "training_loss": float(loss.item()),
            "validation_AP": val_ap,
        })

        # Same early-stopping rule as Deliverable 2.
        if val_ap > best_ap + 1e-5:
            best_ap = val_ap
            best_epoch = epoch
            best_state = copy.deepcopy(current_mlp.state_dict())
            no_improvement = 0
        else:
            no_improvement += 1

        if no_improvement >= 25:
            break

    torch.cuda.synchronize()
    fit_seconds = time.perf_counter() - start

    assert best_state is not None
    current_mlp.load_state_dict(best_state)
    current_mlp.eval()

    with torch.no_grad():
        val_scores = torch.sigmoid(
            current_mlp(mlp_val_x)
        ).cpu().numpy()

    threshold_results = pd.DataFrame([
        {
            "threshold": float(threshold),
            "validation_f1": f1_score(
                mlp_val_y,
                val_scores >= threshold,
                zero_division=0,
            ),
        }
        for threshold in np.round(np.arange(0.05, 1.00, 0.05), 2)
    ])

    best = threshold_results.sort_values(
        ["validation_f1", "threshold"],
        ascending=[False, False],
    ).iloc[0]

    chosen_threshold = float(best["threshold"])
    validation_ap = float(
        average_precision_score(mlp_val_y, val_scores)
    )

    settings = {
        "model": "MLP",
        "seed": seed,
        "selected_epoch": best_epoch,
        "epochs_executed": len(history),
        "threshold": chosen_threshold,
        "validation_AP": validation_ap,
        "validation_F1": float(best["validation_f1"]),
        "positive_class_weight": FIXED_CLASS_WEIGHT,
        "training_with_validation_seconds": fit_seconds,
        "trainable_parameters": sum(
            p.numel() for p in current_mlp.parameters()
        ),
        "training_steps": [1, 30],
        "validation_steps": [31, 34],
        "predictor_columns": local_cols,
        "torch_version": str(torch.__version__),
        "gpu": torch.cuda.get_device_name(0),
        "architecture": [93, 64, 64, 1],
        "dropout": 0.2,
        "learning_rate": 0.001,
        "weight_decay": 1e-4,
        "max_epochs": 200,
        "patience": 25,
        "minimum_AP_gain": 1e-5,
    }

    torch.save(
        {
            "model_state_dict": {
                name: value.detach().cpu()
                for name, value in current_mlp.state_dict().items()
            },
            **settings,
        },
        run_dir / "frozen_model.pt",
    )

    pd.DataFrame(history).to_csv(
        run_dir / "training_history.csv", index=False
    )
    threshold_results.to_csv(
        run_dir / "validation_threshold_search.csv", index=False
    )

    predictions = nodes.loc[
        val_mask, ["txId", "time_step", "y"]
    ].copy()
    predictions["score"] = val_scores
    predictions["prediction"] = (
        val_scores >= chosen_threshold
    ).astype(int)
    predictions.to_csv(
        run_dir / "validation_predictions.csv", index=False
    )

    (run_dir / "run_info.json").write_text(
        json.dumps(settings, indent=2)
    )

    mlp_validation_rows.append({
        "seed": seed,
        "selected_epoch": best_epoch,
        "epochs_executed": len(history),
        "validation_AP": validation_ap,
        "validation_F1": float(best["validation_f1"]),
        "threshold": chosen_threshold,
        "training_seconds": fit_seconds,
    })

    print(f"MLP seed {seed} completed.")

mlp_validation_summary = pd.DataFrame(mlp_validation_rows)
mlp_validation_summary.to_csv(
    MLP_D3_DIR / "validation_summary.csv", index=False
)

print("\nTrainable parameters:", settings["trainable_parameters"])
display(mlp_validation_summary.round(4))
print("Saved to:", MLP_D3_DIR)

MLP seed 1 completed.
MLP seed 2 completed.

Trainable parameters: 10241


,seed,selected_epoch,epochs_executed,validation_AP,validation_F1,threshold,training_seconds
0,1,200,200,0.8840,0.8352,0.8,4.8036
1,2,101,126,0.8706,0.7684,0.7,1.6246


Saved to: /content/drive/MyDrive/Colab Notebooks/AML_GNN_Project/results_deliverable3/mlp


GraphSAGE with seeds 1 and 2

In [5]:
%pip install -q torch-geometric==2.8.1

import torch_geometric
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv

print("PyTorch Geometric:", torch_geometric.__version__)

/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


PyTorch Geometric: 2.8.1


In [6]:
SAGE_D3_DIR = D3_RESULTS / "graphsage"
SAGE_D3_DIR.mkdir(exist_ok=True)

for seed in [1, 2]:
    assert not (SAGE_D3_DIR / f"seed_{seed}").exists(), (
        f"GraphSAGE seed {seed} already exists. Preserve it before rerunning."
    )

# Reuse the scaler fitted on labeled training nodes in Step 4.
assert d3_scaler.feature_names_in_.tolist() == local_cols

def build_d3_graph(first_step, last_step):
    frame = nodes.loc[
        nodes["time_step"].between(first_step, last_step)
    ].copy()

    # Explicit ID mapping preserves feature/edge/target alignment.
    node_index = pd.Series(
        np.arange(len(frame), dtype=np.int64),
        index=frame["txId"].to_numpy(),
    )

    selected_edges = edges.loc[
        edges["txId1"].isin(node_index.index)
        & edges["txId2"].isin(node_index.index)
    ]

    edge_array = np.stack([
        selected_edges["txId1"].map(node_index).to_numpy(
            dtype=np.int64
        ),
        selected_edges["txId2"].map(node_index).to_numpy(
            dtype=np.int64
        ),
    ])

    x_array = d3_scaler.transform(
        frame[local_cols].astype("float32")
    ).astype("float32")

    assert np.isfinite(x_array).all()

    graph = Data(
        x=torch.from_numpy(x_array),
        edge_index=torch.from_numpy(edge_array),
        y=torch.tensor(frame["y"].to_numpy(), dtype=torch.long),
        tx_id=torch.tensor(
            frame["txId"].to_numpy(), dtype=torch.long
        ),
        time_step=torch.tensor(
            frame["time_step"].to_numpy(), dtype=torch.long
        ),
    )
    graph.labeled_mask = graph.y.ge(0)
    return graph.to(D3_DEVICE)

d3_train_graph = build_d3_graph(1, 30)
d3_val_graph = build_d3_graph(31, 34)

assert d3_train_graph.num_nodes == 123287
assert d3_train_graph.num_edges == 142784
assert d3_val_graph.num_nodes == 12978
assert d3_val_graph.num_edges == 14059

assert int(d3_train_graph.labeled_mask.sum().item()) == 26905
assert int(d3_val_graph.labeled_mask.sum().item()) == 2989

# Confirm validation predictions will match the original row order.
np.testing.assert_array_equal(
    d3_val_graph.tx_id[d3_val_graph.labeled_mask].cpu().numpy(),
    nodes.loc[val_mask, "txId"].to_numpy(),
)

print(
    "Training graph:",
    d3_train_graph.num_nodes, "nodes,",
    d3_train_graph.num_edges, "edges"
)
print(
    "Validation graph:",
    d3_val_graph.num_nodes, "nodes,",
    d3_val_graph.num_edges, "edges"
)

joblib.dump(d3_scaler, SAGE_D3_DIR / "training_scaler.joblib")

class TransactionGraphSAGE(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = SAGEConv(93, 64, aggr="mean")
        self.conv2 = SAGEConv(64, 64, aggr="mean")
        self.dropout = nn.Dropout(0.2)
        self.output = nn.Linear(64, 1)

    def forward(self, x, edge_index):
        x = self.dropout(torch.relu(self.conv1(x, edge_index)))
        x = self.dropout(torch.relu(self.conv2(x, edge_index)))
        return self.output(x).squeeze(-1)

sage_val_y = (
    d3_val_graph.y[d3_val_graph.labeled_mask].cpu().numpy()
)
sage_validation_rows = []

for seed in [1, 2]:
    set_d3_seed(seed)
    run_dir = SAGE_D3_DIR / f"seed_{seed}"
    run_dir.mkdir()

    current_sage = TransactionGraphSAGE().to(D3_DEVICE)
    optimizer = torch.optim.Adam(
        current_sage.parameters(), lr=0.001, weight_decay=1e-4
    )
    criterion = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor(
            FIXED_CLASS_WEIGHT,
            dtype=torch.float32,
            device=D3_DEVICE,
        )
    )

    best_ap = -np.inf
    best_epoch = 0
    best_state = None
    no_improvement = 0
    history = []

    torch.cuda.synchronize()
    start = time.perf_counter()

    for epoch in range(1, 201):
        current_sage.train()
        optimizer.zero_grad(set_to_none=True)

        train_logits = current_sage(
            d3_train_graph.x, d3_train_graph.edge_index
        )

        # Unknown targets never enter supervised loss.
        loss = criterion(
            train_logits[d3_train_graph.labeled_mask],
            d3_train_graph.y[
                d3_train_graph.labeled_mask
            ].float(),
        )

        assert torch.isfinite(loss), "Non-finite training loss."
        loss.backward()
        optimizer.step()

        current_sage.eval()
        with torch.no_grad():
            val_logits = current_sage(
                d3_val_graph.x, d3_val_graph.edge_index
            )
            val_scores = torch.sigmoid(
                val_logits[d3_val_graph.labeled_mask]
            ).cpu().numpy()

        assert np.isfinite(val_scores).all()
        val_ap = float(
            average_precision_score(sage_val_y, val_scores)
        )

        history.append({
            "epoch": epoch,
            "training_loss": float(loss.item()),
            "validation_AP": val_ap,
        })

        if val_ap > best_ap + 1e-5:
            best_ap = val_ap
            best_epoch = epoch
            best_state = copy.deepcopy(current_sage.state_dict())
            no_improvement = 0
        else:
            no_improvement += 1

        if no_improvement >= 25:
            break

    torch.cuda.synchronize()
    fit_seconds = time.perf_counter() - start

    assert best_state is not None
    current_sage.load_state_dict(best_state)
    current_sage.eval()

    with torch.no_grad():
        val_logits = current_sage(
            d3_val_graph.x, d3_val_graph.edge_index
        )
        val_scores = torch.sigmoid(
            val_logits[d3_val_graph.labeled_mask]
        ).cpu().numpy()

    threshold_results = pd.DataFrame([
        {
            "threshold": float(threshold),
            "validation_f1": f1_score(
                sage_val_y,
                val_scores >= threshold,
                zero_division=0,
            ),
        }
        for threshold in np.round(np.arange(0.05, 1.00, 0.05), 2)
    ])

    best = threshold_results.sort_values(
        ["validation_f1", "threshold"],
        ascending=[False, False],
    ).iloc[0]

    chosen_threshold = float(best["threshold"])
    validation_ap = float(
        average_precision_score(sage_val_y, val_scores)
    )

    settings = {
        "model": "GraphSAGE",
        "seed": seed,
        "selected_epoch": best_epoch,
        "epochs_executed": len(history),
        "threshold": chosen_threshold,
        "validation_AP": validation_ap,
        "validation_F1": float(best["validation_f1"]),
        "positive_class_weight": FIXED_CLASS_WEIGHT,
        "training_with_validation_seconds": fit_seconds,
        "trainable_parameters": sum(
            p.numel() for p in current_sage.parameters()
        ),
        "training_steps": [1, 30],
        "validation_steps": [31, 34],
        "predictor_columns": local_cols,
        "torch_version": str(torch.__version__),
        "pyg_version": str(torch_geometric.__version__),
        "gpu": torch.cuda.get_device_name(0),
        "edge_direction": "incoming_original_directed",
        "architecture": [93, 64, 64, 1],
        "aggregation": "mean",
        "dropout": 0.2,
        "learning_rate": 0.001,
        "weight_decay": 1e-4,
        "max_epochs": 200,
        "patience": 25,
        "minimum_AP_gain": 1e-5,
    }

    torch.save(
        {
            "model_state_dict": {
                name: value.detach().cpu()
                for name, value in current_sage.state_dict().items()
            },
            **settings,
        },
        run_dir / "frozen_model.pt",
    )

    pd.DataFrame(history).to_csv(
        run_dir / "training_history.csv", index=False
    )
    threshold_results.to_csv(
        run_dir / "validation_threshold_search.csv", index=False
    )

    predictions = nodes.loc[
        val_mask, ["txId", "time_step", "y"]
    ].copy()
    predictions["score"] = val_scores
    predictions["prediction"] = (
        val_scores >= chosen_threshold
    ).astype(int)
    predictions.to_csv(
        run_dir / "validation_predictions.csv", index=False
    )

    (run_dir / "run_info.json").write_text(
        json.dumps(settings, indent=2)
    )

    sage_validation_rows.append({
        "seed": seed,
        "selected_epoch": best_epoch,
        "epochs_executed": len(history),
        "validation_AP": validation_ap,
        "validation_F1": float(best["validation_f1"]),
        "threshold": chosen_threshold,
        "training_seconds": fit_seconds,
    })

    print(f"GraphSAGE seed {seed} completed.")

sage_validation_summary = pd.DataFrame(sage_validation_rows)
sage_validation_summary.to_csv(
    SAGE_D3_DIR / "validation_summary.csv", index=False
)

print("\nTrainable parameters:", settings["trainable_parameters"])
display(sage_validation_summary.round(4))
print("Saved to:", SAGE_D3_DIR)

Training graph: 123287 nodes, 142784 edges
Validation graph: 12978 nodes, 14059 edges
GraphSAGE seed 1 completed.
GraphSAGE seed 2 completed.

Trainable parameters: 20289


,seed,selected_epoch,epochs_executed,validation_AP,validation_F1,threshold,training_seconds
0,1,199,200,0.8935,0.8543,0.90,7.4909
1,2,60,85,0.7989,0.8154,0.85,2.1459


Saved to: /content/drive/MyDrive/Colab Notebooks/AML_GNN_Project/results_deliverable3/graphsage


Evaluate the frozen models

In [7]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    confusion_matrix,
)

def evaluate_d3_predictions(predictions, model_name, policy, seed):
    """Evaluate each snapshot separately on known labels."""
    rows = []

    assert not predictions["txId"].duplicated().any()
    assert predictions["y"].isin([0, 1]).all()
    assert np.isfinite(predictions["score"]).all()
    assert predictions["score"].between(0, 1).all()

    for step, subset in predictions.groupby("time_step", sort=True):
        y = subset["y"].to_numpy()
        scores = subset["score"].to_numpy()
        predicted = subset["prediction"].to_numpy()

        tn, fp, fn, tp = confusion_matrix(
            y, predicted, labels=[0, 1]
        ).ravel()

        row = {
            "model": model_name,
            "policy": policy,
            "seed": int(seed),
            "time_step": int(step),
            "n_labeled": len(subset),
            "n_illicit": int(y.sum()),
            "illicit_prevalence": float(y.mean()),
            "average_precision": (
                float(average_precision_score(y, scores))
                if len(np.unique(y)) == 2 else np.nan
            ),
            "precision": precision_score(
                y, predicted, zero_division=0
            ),
            "recall": recall_score(
                y, predicted, zero_division=0
            ),
            "f1": f1_score(y, predicted, zero_division=0),
            "tn": int(tn),
            "fp": int(fp),
            "fn": int(fn),
            "tp": int(tp),
        }

        # Stable ranking preserves the original order for tied scores.
        order = np.argsort(-scores, kind="stable")

        for budget in [1, 5, 10, 20]:
            k = max(1, int(np.ceil(len(y) * budget / 100)))
            top_labels = y[order[:k]]

            row[f"review_k_{budget}pct"] = k
            row[f"precision_at_{budget}pct"] = float(
                top_labels.mean()
            )
            row[f"recall_at_{budget}pct"] = (
                float(top_labels.sum() / y.sum())
                if y.sum() > 0 else np.nan
            )

        # Compatible with the Deliverable 2 metric name.
        row["review_k"] = row["review_k_10pct"]
        rows.append(row)

    return pd.DataFrame(rows)


test_mask = (
    nodes["time_step"].between(35, 49)
    & nodes["y"].ge(0)
)

test_reference = nodes.loc[
    test_mask, ["txId", "time_step", "y"]
].copy()

assert len(test_reference) == 16670

model_directories = {
    "XGBoost": XGB_D3_DIR,
    "MLP": MLP_D3_DIR,
    "GraphSAGE": SAGE_D3_DIR,
}

# Check every output path before starting.
for model_name, parent in model_directories.items():
    for seed in [1, 2]:
        path = parent / f"seed_{seed}" / "frozen_test_predictions.csv"
        assert not path.exists(), (
            f"Existing evaluation found: {path}. Preserve it before rerunning."
        )

# Verify the saved neural preprocessing matches the current scaler.
for parent in [MLP_D3_DIR, SAGE_D3_DIR]:
    saved_scaler = joblib.load(parent / "training_scaler.joblib")
    assert saved_scaler.feature_names_in_.tolist() == local_cols
    np.testing.assert_array_equal(
        saved_scaler.mean_, d3_scaler.mean_
    )
    np.testing.assert_array_equal(
        saved_scaler.scale_, d3_scaler.scale_
    )

# Build test inputs without fitting any preprocessing.
test_local = nodes.loc[
    test_mask, local_cols
].astype("float32")

mlp_test_x = torch.from_numpy(
    d3_scaler.transform(test_local).astype("float32")
).to(D3_DEVICE)

d3_test_graph = build_d3_graph(35, 49)
assert d3_test_graph.num_nodes == 67504
assert d3_test_graph.num_edges == 77512

np.testing.assert_array_equal(
    d3_test_graph.tx_id[
        d3_test_graph.labeled_mask
    ].cpu().numpy(),
    test_reference["txId"].to_numpy(),
)

all_frozen_metrics = []

for model_name, parent in model_directories.items():
    for seed in [1, 2]:
        run_dir = parent / f"seed_{seed}"
        settings = json.loads(
            (run_dir / "run_info.json").read_text()
        )
        threshold = float(settings["threshold"])

        assert settings["predictor_columns"] == local_cols

        if model_name == "XGBoost":
            booster = xgb.Booster()
            booster.load_model(str(run_dir / "frozen_model.json"))
            booster.set_param({"device": "cuda"})

            scores = booster.predict(xgb.DMatrix(test_local))

        else:
            checkpoint = torch.load(
                run_dir / "frozen_model.pt",
                map_location="cpu",
                weights_only=True,
            )

            if model_name == "MLP":
                fitted_model = TransactionMLP().to(D3_DEVICE)
            else:
                fitted_model = TransactionGraphSAGE().to(D3_DEVICE)

            fitted_model.load_state_dict(
                checkpoint["model_state_dict"]
            )
            fitted_model.eval()

            with torch.no_grad():
                if model_name == "MLP":
                    logits = fitted_model(mlp_test_x)
                else:
                    logits = fitted_model(
                        d3_test_graph.x,
                        d3_test_graph.edge_index,
                    )[d3_test_graph.labeled_mask]

                scores = torch.sigmoid(logits).cpu().numpy()

        assert len(scores) == len(test_reference)

        predictions = test_reference.copy()
        predictions["score"] = scores
        predictions["prediction"] = (
            scores >= threshold
        ).astype(int)
        predictions["model"] = model_name
        predictions["policy"] = "frozen"
        predictions["seed"] = seed

        metrics = evaluate_d3_predictions(
            predictions, model_name, "frozen", seed
        )

        assert len(metrics) == 15

        predictions.to_csv(
            run_dir / "frozen_test_predictions.csv", index=False
        )
        metrics.to_csv(
            run_dir / "frozen_metrics_by_step.csv", index=False
        )

        all_frozen_metrics.append(metrics)
        print(
            f"{model_name}, seed {seed}: "
            f"{len(predictions):,} transactions scored."
        )

d3_frozen_metrics = pd.concat(
    all_frozen_metrics, ignore_index=True
)

assert len(d3_frozen_metrics) == 90
assert not d3_frozen_metrics.duplicated(
    ["model", "policy", "seed", "time_step"]
).any()

d3_frozen_metrics.to_csv(
    D3_RESULTS / "frozen_metrics_seeds1_2.csv",
    index=False,
)

# Average snapshot metrics, matching the Deliverable 2 approach.
phase_metrics = d3_frozen_metrics.copy()
phase_metrics["phase"] = np.select(
    [
        phase_metrics["time_step"].between(35, 42),
        phase_metrics["time_step"].eq(43),
    ],
    ["before_shift", "transition"],
    default="after_shift",
)

frozen_summary = (
    phase_metrics[
        phase_metrics["phase"] != "transition"
    ]
    .groupby(["model", "seed", "phase"], as_index=False)
    .agg(
        mean_AP=("average_precision", "mean"),
        mean_F1=("f1", "mean"),
        mean_recall=("recall", "mean"),
    )
)

frozen_summary.to_csv(
    D3_RESULTS / "frozen_phase_summary_seeds1_2.csv",
    index=False,
)

print("\nVerified frozen model-step results:", len(d3_frozen_metrics))
print("Expected: 90")
display(frozen_summary.round(4))

XGBoost, seed 1: 16,670 transactions scored.
XGBoost, seed 2: 16,670 transactions scored.
MLP, seed 1: 16,670 transactions scored.
MLP, seed 2: 16,670 transactions scored.
GraphSAGE, seed 1: 16,670 transactions scored.
GraphSAGE, seed 2: 16,670 transactions scored.

Verified frozen model-step results: 90
Expected: 90


,model,seed,phase,mean_AP,mean_F1,mean_recall
0,GraphSAGE,1,after_shift,0.1536,0.0174,0.0933
1,GraphSAGE,1,before_shift,0.7159,0.6503,0.6796
2,GraphSAGE,2,after_shift,0.0686,0.0171,0.0433
3,GraphSAGE,2,before_shift,0.6791,0.6287,0.6542
4,MLP,1,after_shift,0.0968,0.0108,0.0933
5,MLP,1,before_shift,0.7456,0.6318,0.8635
6,MLP,2,after_shift,0.1326,0.0247,0.1550
7,MLP,2,before_shift,0.8436,0.5949,0.8763
8,XGBoost,1,after_shift,0.0789,0.0242,0.1240
9,XGBoost,1,before_shift,0.9040,0.7268,0.8772


Run XGBoost adaptation w/ final summary table

In [8]:
ADAPTATION_POLICIES_D3 = ["expanding", "rolling", "recency"]
ROLLING_WINDOW_D3 = 8
RECENCY_HALF_LIFE_D3 = 4

def select_d3_history(score_step, policy):
    times = nodes["time_step"]
    eligible = times.lt(score_step) & nodes["y"].ge(0)

    if policy == "rolling":
        eligible &= times.ge(score_step - ROLLING_WINDOW_D3)
    elif policy not in ["expanding", "recency"]:
        raise ValueError(f"Unknown policy: {policy}")

    history = nodes.loc[eligible].copy()

    assert not history.empty
    assert history["time_step"].max() < score_step
    assert history["y"].nunique() == 2

    weights = np.ones(len(history), dtype=np.float32)

    if policy == "recency":
        age = (
            score_step - 1
            - history["time_step"].to_numpy()
        )
        weights = np.exp2(
            -age / RECENCY_HALF_LIFE_D3
        ).astype(np.float32)
        weights /= weights.mean()

    assert np.isfinite(weights).all()
    assert (weights > 0).all()
    assert np.isclose(weights.mean(), 1.0, atol=1e-6)

    return history, weights


# Check the temporal boundaries before any refitting.
history_checks = []

for policy in ADAPTATION_POLICIES_D3:
    history, weights = select_d3_history(44, policy)
    history_checks.append({
        "policy": policy,
        "prediction_step": 44,
        "first_training_step": int(history["time_step"].min()),
        "last_training_step": int(history["time_step"].max()),
        "training_labels": len(history),
        "mean_weight": float(weights.mean()),
    })

display(pd.DataFrame(history_checks))

# Protect existing adaptation runs.
for seed in [1, 2]:
    output_dir = XGB_D3_DIR / f"seed_{seed}" / "adaptation"
    assert not output_dir.exists(), (
        f"{output_dir} already exists. Preserve it before rerunning."
    )

all_xgb_adaptation_metrics = []

for seed in [1, 2]:
    run_dir = XGB_D3_DIR / f"seed_{seed}"
    output_dir = run_dir / "adaptation"
    output_dir.mkdir()

    settings = json.loads(
        (run_dir / "run_info.json").read_text()
    )
    threshold = float(settings["threshold"])
    class_weight = float(settings["positive_class_weight"])

    protocol = {
        "model": "XGBoost",
        "seed": seed,
        "policies": ADAPTATION_POLICIES_D3,
        "rolling_window": ROLLING_WINDOW_D3,
        "recency_half_life": RECENCY_HALF_LIFE_D3,
        "simulated_label_delay_steps": 1,
        "threshold": threshold,
        "positive_class_weight": class_weight,
        "refit_from_scratch": True,
        "test_steps": [35, 49],
        "predictor_columns": local_cols,
    }
    (output_dir / "protocol.json").write_text(
        json.dumps(protocol, indent=2)
    )

    seed_metrics = []
    seed_predictions = []
    timing_rows = []

    for policy in ADAPTATION_POLICIES_D3:
        policy_dir = output_dir / policy
        policy_dir.mkdir()

        for score_step in range(35, 50):
            history, weights = select_d3_history(
                score_step, policy
            )

            score_rows = nodes.loc[
                nodes["time_step"].eq(score_step)
                & nodes["y"].ge(0)
            ].copy()

            assert not score_rows.empty

            # Same model settings as the frozen run.
            # Only training history/sample weights change.
            adaptive_model = xgb.XGBClassifier(
                **settings["parameters"],
                tree_method="hist",
                device="cuda",
                objective="binary:logistic",
                eval_metric="logloss",
                scale_pos_weight=class_weight,
                random_state=seed,
                n_jobs=2,
            )

            torch.cuda.synchronize()
            start = time.perf_counter()

            adaptive_model.fit(
                history[local_cols].astype("float32"),
                history["y"],
                sample_weight=weights,
            )

            torch.cuda.synchronize()
            refit_seconds = time.perf_counter() - start

            # Generate scores before using current-step targets.
            scores = adaptive_model.get_booster().predict(
                xgb.DMatrix(
                    score_rows[local_cols].astype("float32")
                )
            )
            assert np.isfinite(scores).all()

            predictions = score_rows[
                ["txId", "time_step", "y"]
            ].copy()
            predictions["score"] = scores
            predictions["prediction"] = (
                scores >= threshold
            ).astype(int)
            predictions["model"] = "XGBoost"
            predictions["policy"] = policy
            predictions["seed"] = seed

            metrics = evaluate_d3_predictions(
                predictions, "XGBoost", policy, seed
            )

            # Save each completed step immediately.
            predictions.to_csv(
                policy_dir / f"step_{score_step}_predictions.csv",
                index=False,
            )
            metrics.to_csv(
                policy_dir / f"step_{score_step}_metrics.csv",
                index=False,
            )

            timing_rows.append({
                "model": "XGBoost",
                "seed": seed,
                "policy": policy,
                "score_step": score_step,
                "first_training_step": int(
                    history["time_step"].min()
                ),
                "last_training_step": int(
                    history["time_step"].max()
                ),
                "training_labels": len(history),
                "training_illicit": int(history["y"].sum()),
                "refit_seconds": refit_seconds,
            })

            seed_metrics.append(metrics)
            seed_predictions.append(predictions)

        print(f"Seed {seed}: {policy} completed.")

    seed_metrics_df = pd.concat(seed_metrics, ignore_index=True)
    seed_predictions_df = pd.concat(
        seed_predictions, ignore_index=True
    )

    assert len(seed_metrics_df) == 45
    assert len(seed_predictions_df) == 16670 * 3
    assert not seed_predictions_df.duplicated(
        ["policy", "txId"]
    ).any()

    seed_metrics_df.to_csv(
        output_dir / "metrics_by_step.csv", index=False
    )
    seed_predictions_df.to_csv(
        output_dir / "test_predictions.csv", index=False
    )
    pd.DataFrame(timing_rows).to_csv(
        output_dir / "refit_timing.csv", index=False
    )

    all_xgb_adaptation_metrics.append(seed_metrics_df)

xgb_d3_adaptation_metrics = pd.concat(
    all_xgb_adaptation_metrics, ignore_index=True
)

assert len(xgb_d3_adaptation_metrics) == 90
assert not xgb_d3_adaptation_metrics.duplicated(
    ["model", "policy", "seed", "time_step"]
).any()

xgb_d3_adaptation_metrics.to_csv(
    D3_RESULTS / "xgb_adaptation_metrics_seeds1_2.csv",
    index=False,
)

post_shift_summary = (
    xgb_d3_adaptation_metrics[
        xgb_d3_adaptation_metrics["time_step"].between(44, 49)
    ]
    .groupby(["seed", "policy"], as_index=False)
    .agg(
        mean_AP=("average_precision", "mean"),
        mean_F1=("f1", "mean"),
        mean_recall=("recall", "mean"),
    )
)

post_shift_summary.to_csv(
    XGB_D3_DIR / "adaptation_post_shift_summary.csv",
    index=False,
)

print("\nVerified XGBoost adaptation results:",
      len(xgb_d3_adaptation_metrics))
print("Expected: 90")
display(post_shift_summary.round(4))

,policy,prediction_step,first_training_step,last_training_step,training_labels,mean_weight
0,expanding,44,1,43,41247,1.0
1,rolling,44,36,43,10012,1.0
2,recency,44,1,43,41247,1.0


Seed 1: expanding completed.
Seed 1: rolling completed.
Seed 1: recency completed.
Seed 2: expanding completed.
Seed 2: rolling completed.
Seed 2: recency completed.

Verified XGBoost adaptation results: 90
Expected: 90


,seed,policy,mean_AP,mean_F1,mean_recall
0,1,expanding,0.1983,0.0872,0.1504
1,1,recency,0.4053,0.2685,0.2764
2,1,rolling,0.3164,0.2120,0.1739
3,2,expanding,0.1983,0.0872,0.1504
4,2,recency,0.4053,0.2685,0.2764
5,2,rolling,0.3164,0.2120,0.1739


Run MLP adaptation w/ final summary table

In [9]:
# Reuse preprocessing fitted only on the original training partition.
mlp_adapt_scaler = joblib.load(
    MLP_D3_DIR / "training_scaler.joblib"
)
assert mlp_adapt_scaler.feature_names_in_.tolist() == local_cols

# Protect existing adaptation runs before starting.
for seed in [1, 2]:
    output_dir = MLP_D3_DIR / f"seed_{seed}" / "adaptation"
    assert not output_dir.exists(), (
        f"{output_dir} already exists. Preserve it before rerunning."
    )

all_mlp_adaptation_metrics = []

for seed in [1, 2]:
    run_dir = MLP_D3_DIR / f"seed_{seed}"
    output_dir = run_dir / "adaptation"
    output_dir.mkdir()

    settings = json.loads(
        (run_dir / "run_info.json").read_text()
    )

    threshold = float(settings["threshold"])
    epochs_per_refit = int(settings["selected_epoch"])
    class_weight = float(settings["positive_class_weight"])

    assert settings["predictor_columns"] == local_cols
    assert 1 <= epochs_per_refit <= 200

    protocol = {
        "model": "MLP",
        "seed": seed,
        "policies": ADAPTATION_POLICIES_D3,
        "rolling_window": ROLLING_WINDOW_D3,
        "recency_half_life": RECENCY_HALF_LIFE_D3,
        "simulated_label_delay_steps": 1,
        "threshold": threshold,
        "epochs_per_refit": epochs_per_refit,
        "positive_class_weight": class_weight,
        "refit_from_scratch": True,
        "scaler_refitted": False,
        "test_early_stopping": False,
        "test_steps": [35, 49],
        "predictor_columns": local_cols,
    }
    (output_dir / "protocol.json").write_text(
        json.dumps(protocol, indent=2)
    )

    print(
        f"\nSeed {seed}: {epochs_per_refit} epochs per refit, "
        f"threshold {threshold:.2f}"
    )

    seed_metrics = []
    seed_predictions = []
    timing_rows = []

    for policy in ADAPTATION_POLICIES_D3:
        policy_dir = output_dir / policy
        policy_dir.mkdir()

        for score_step in range(35, 50):
            history, weights = select_d3_history(
                score_step, policy
            )

            score_rows = nodes.loc[
                nodes["time_step"].eq(score_step)
                & nodes["y"].ge(0)
            ].copy()

            assert not score_rows.empty

            train_array = mlp_adapt_scaler.transform(
                history[local_cols].astype("float32")
            ).astype("float32")

            score_array = mlp_adapt_scaler.transform(
                score_rows[local_cols].astype("float32")
            ).astype("float32")

            assert np.isfinite(train_array).all()
            assert np.isfinite(score_array).all()

            train_x = torch.from_numpy(
                train_array
            ).to(D3_DEVICE)
            train_y = torch.tensor(
                history["y"].to_numpy(),
                dtype=torch.float32,
                device=D3_DEVICE,
            )
            train_weights = torch.tensor(
                weights,
                dtype=torch.float32,
                device=D3_DEVICE,
            )
            score_x = torch.from_numpy(
                score_array
            ).to(D3_DEVICE)

            # Reset initialization for every policy and step.
            set_d3_seed(seed)
            adaptive_mlp = TransactionMLP().to(D3_DEVICE)

            optimizer = torch.optim.Adam(
                adaptive_mlp.parameters(),
                lr=settings["learning_rate"],
                weight_decay=settings["weight_decay"],
            )
            criterion = nn.BCEWithLogitsLoss(
                pos_weight=torch.tensor(
                    class_weight,
                    dtype=torch.float32,
                    device=D3_DEVICE,
                ),
                reduction="none",
            )

            torch.cuda.synchronize()
            start = time.perf_counter()

            for epoch in range(epochs_per_refit):
                adaptive_mlp.train()
                optimizer.zero_grad(set_to_none=True)

                individual_losses = criterion(
                    adaptive_mlp(train_x), train_y
                )
                loss = (
                    individual_losses * train_weights
                ).sum() / train_weights.sum()

                assert torch.isfinite(loss), (
                    f"Non-finite loss: seed {seed}, "
                    f"{policy}, step {score_step}"
                )

                loss.backward()
                optimizer.step()

            torch.cuda.synchronize()
            refit_seconds = time.perf_counter() - start

            # Predict before using current-step labels in evaluation.
            adaptive_mlp.eval()
            with torch.no_grad():
                scores = torch.sigmoid(
                    adaptive_mlp(score_x)
                ).cpu().numpy()

            assert np.isfinite(scores).all()

            predictions = score_rows[
                ["txId", "time_step", "y"]
            ].copy()
            predictions["score"] = scores
            predictions["prediction"] = (
                scores >= threshold
            ).astype(int)
            predictions["model"] = "MLP"
            predictions["policy"] = policy
            predictions["seed"] = seed

            metrics = evaluate_d3_predictions(
                predictions, "MLP", policy, seed
            )

            predictions.to_csv(
                policy_dir / f"step_{score_step}_predictions.csv",
                index=False,
            )
            metrics.to_csv(
                policy_dir / f"step_{score_step}_metrics.csv",
                index=False,
            )

            timing_rows.append({
                "model": "MLP",
                "seed": seed,
                "policy": policy,
                "score_step": score_step,
                "first_training_step": int(
                    history["time_step"].min()
                ),
                "last_training_step": int(
                    history["time_step"].max()
                ),
                "training_labels": len(history),
                "training_illicit": int(history["y"].sum()),
                "epochs_per_refit": epochs_per_refit,
                "final_training_loss": float(loss.item()),
                "refit_seconds": refit_seconds,
            })

            seed_metrics.append(metrics)
            seed_predictions.append(predictions)

        print(f"Seed {seed}: {policy} completed.")

    seed_metrics_df = pd.concat(seed_metrics, ignore_index=True)
    seed_predictions_df = pd.concat(
        seed_predictions, ignore_index=True
    )

    assert len(seed_metrics_df) == 45
    assert len(seed_predictions_df) == 16670 * 3
    assert not seed_predictions_df.duplicated(
        ["policy", "txId"]
    ).any()

    # Every policy must evaluate the same transactions and targets.
    reference = test_reference.sort_values(
        "txId"
    ).reset_index(drop=True)

    for policy in ADAPTATION_POLICIES_D3:
        observed = (
            seed_predictions_df.loc[
                seed_predictions_df["policy"].eq(policy),
                ["txId", "time_step", "y"],
            ]
            .sort_values("txId")
            .reset_index(drop=True)
        )
        pd.testing.assert_frame_equal(reference, observed)

    seed_metrics_df.to_csv(
        output_dir / "metrics_by_step.csv", index=False
    )
    seed_predictions_df.to_csv(
        output_dir / "test_predictions.csv", index=False
    )
    pd.DataFrame(timing_rows).to_csv(
        output_dir / "refit_timing.csv", index=False
    )

    all_mlp_adaptation_metrics.append(seed_metrics_df)

mlp_d3_adaptation_metrics = pd.concat(
    all_mlp_adaptation_metrics, ignore_index=True
)

assert len(mlp_d3_adaptation_metrics) == 90
assert not mlp_d3_adaptation_metrics.duplicated(
    ["model", "policy", "seed", "time_step"]
).any()

mlp_d3_adaptation_metrics.to_csv(
    D3_RESULTS / "mlp_adaptation_metrics_seeds1_2.csv",
    index=False,
)

mlp_post_shift_summary = (
    mlp_d3_adaptation_metrics[
        mlp_d3_adaptation_metrics["time_step"].between(44, 49)
    ]
    .groupby(["seed", "policy"], as_index=False)
    .agg(
        mean_AP=("average_precision", "mean"),
        mean_F1=("f1", "mean"),
        mean_recall=("recall", "mean"),
    )
)

mlp_post_shift_summary.to_csv(
    MLP_D3_DIR / "adaptation_post_shift_summary.csv",
    index=False,
)

print("\nVerified MLP adaptation results:",
      len(mlp_d3_adaptation_metrics))
print("Expected: 90")
display(mlp_post_shift_summary.round(4))


Seed 1: 200 epochs per refit, threshold 0.80
Seed 1: expanding completed.
Seed 1: rolling completed.
Seed 1: recency completed.

Seed 2: 101 epochs per refit, threshold 0.70
Seed 2: expanding completed.
Seed 2: rolling completed.
Seed 2: recency completed.

Verified MLP adaptation results: 90
Expected: 90


,seed,policy,mean_AP,mean_F1,mean_recall
0,1,expanding,0.0712,0.0130,0.0099
1,1,recency,0.1698,0.0796,0.0489
2,1,rolling,0.1607,0.0632,0.0370
3,2,expanding,0.0571,0.0106,0.0099
4,2,recency,0.1009,0.0342,0.0222
5,2,rolling,0.0932,0.0433,0.0291


Run GraphSAGE adaptation w/ final summary table

In [10]:
sage_adapt_scaler = joblib.load(
    SAGE_D3_DIR / "training_scaler.joblib"
)

assert sage_adapt_scaler.feature_names_in_.tolist() == local_cols
np.testing.assert_array_equal(
    sage_adapt_scaler.mean_, d3_scaler.mean_
)
np.testing.assert_array_equal(
    sage_adapt_scaler.scale_, d3_scaler.scale_
)

# Protect existing adaptation runs.
for seed in [1, 2]:
    output_dir = SAGE_D3_DIR / f"seed_{seed}" / "adaptation"
    assert not output_dir.exists(), (
        f"{output_dir} already exists. Preserve it before rerunning."
    )

# Each scoring graph contains only its own snapshot.
# No scaler fitting or training occurs here.
sage_score_graphs = {
    step: build_d3_graph(step, step)
    for step in range(35, 50)
}

all_sage_adaptation_metrics = []

for seed in [1, 2]:
    run_dir = SAGE_D3_DIR / f"seed_{seed}"
    output_dir = run_dir / "adaptation"
    output_dir.mkdir()

    settings = json.loads(
        (run_dir / "run_info.json").read_text()
    )

    threshold = float(settings["threshold"])
    epochs_per_refit = int(settings["selected_epoch"])
    class_weight = float(settings["positive_class_weight"])

    assert settings["predictor_columns"] == local_cols
    assert 1 <= epochs_per_refit <= 200

    protocol = {
        "model": "GraphSAGE",
        "seed": seed,
        "policies": ADAPTATION_POLICIES_D3,
        "rolling_window": ROLLING_WINDOW_D3,
        "recency_half_life": RECENCY_HALF_LIFE_D3,
        "simulated_label_delay_steps": 1,
        "threshold": threshold,
        "epochs_per_refit": epochs_per_refit,
        "positive_class_weight": class_weight,
        "refit_from_scratch": True,
        "scaler_refitted": False,
        "test_early_stopping": False,
        "edge_direction": "incoming_original_directed",
        "unknown_targets_masked": True,
        "test_steps": [35, 49],
        "predictor_columns": local_cols,
    }

    (output_dir / "protocol.json").write_text(
        json.dumps(protocol, indent=2)
    )

    print(
        f"\nSeed {seed}: {epochs_per_refit} epochs per refit, "
        f"threshold {threshold:.2f}"
    )

    seed_metrics = []
    seed_predictions = []
    timing_rows = []

    for policy in ADAPTATION_POLICIES_D3:
        policy_dir = output_dir / policy
        policy_dir.mkdir()

        for score_step in range(35, 50):
            history, weights = select_d3_history(
                score_step, policy
            )

            first_step = (
                score_step - ROLLING_WINDOW_D3
                if policy == "rolling" else 1
            )

            train_graph = build_d3_graph(
                first_step, score_step - 1
            )
            score_graph = sage_score_graphs[score_step]
            train_known = train_graph.labeled_mask
            score_known = score_graph.labeled_mask

            assert int(train_graph.time_step.max().item()) < score_step
            assert int(train_graph.time_step.min().item()) >= first_step

            # Ensure sample weights match labeled graph-node order.
            np.testing.assert_array_equal(
                train_graph.tx_id[train_known].cpu().numpy(),
                history["txId"].to_numpy(),
            )
            np.testing.assert_array_equal(
                train_graph.y[train_known].cpu().numpy(),
                history["y"].to_numpy(),
            )

            score_rows = nodes.loc[
                nodes["time_step"].eq(score_step)
                & nodes["y"].ge(0)
            ].copy()

            np.testing.assert_array_equal(
                score_graph.tx_id[score_known].cpu().numpy(),
                score_rows["txId"].to_numpy(),
            )

            train_weights = torch.tensor(
                weights,
                dtype=torch.float32,
                device=D3_DEVICE,
            )

            set_d3_seed(seed)
            adaptive_sage = TransactionGraphSAGE().to(D3_DEVICE)

            optimizer = torch.optim.Adam(
                adaptive_sage.parameters(),
                lr=settings["learning_rate"],
                weight_decay=settings["weight_decay"],
            )
            criterion = nn.BCEWithLogitsLoss(
                pos_weight=torch.tensor(
                    class_weight,
                    dtype=torch.float32,
                    device=D3_DEVICE,
                ),
                reduction="none",
            )

            torch.cuda.synchronize()
            start = time.perf_counter()

            for epoch in range(epochs_per_refit):
                adaptive_sage.train()
                optimizer.zero_grad(set_to_none=True)

                logits = adaptive_sage(
                    train_graph.x,
                    train_graph.edge_index,
                )

                individual_losses = criterion(
                    logits[train_known],
                    train_graph.y[train_known].float(),
                )

                loss = (
                    individual_losses * train_weights
                ).sum() / train_weights.sum()

                assert torch.isfinite(loss), (
                    f"Non-finite loss: seed {seed}, "
                    f"{policy}, step {score_step}"
                )

                loss.backward()
                optimizer.step()

            torch.cuda.synchronize()
            refit_seconds = time.perf_counter() - start

            adaptive_sage.eval()
            with torch.no_grad():
                score_logits = adaptive_sage(
                    score_graph.x,
                    score_graph.edge_index,
                )
                scores = torch.sigmoid(
                    score_logits[score_known]
                ).cpu().numpy()

            assert np.isfinite(scores).all()

            predictions = score_rows[
                ["txId", "time_step", "y"]
            ].copy()
            predictions["score"] = scores
            predictions["prediction"] = (
                scores >= threshold
            ).astype(int)
            predictions["model"] = "GraphSAGE"
            predictions["policy"] = policy
            predictions["seed"] = seed

            metrics = evaluate_d3_predictions(
                predictions, "GraphSAGE", policy, seed
            )

            predictions.to_csv(
                policy_dir / f"step_{score_step}_predictions.csv",
                index=False,
            )
            metrics.to_csv(
                policy_dir / f"step_{score_step}_metrics.csv",
                index=False,
            )

            timing_rows.append({
                "model": "GraphSAGE",
                "seed": seed,
                "policy": policy,
                "score_step": score_step,
                "first_training_step": first_step,
                "last_training_step": score_step - 1,
                "training_nodes": train_graph.num_nodes,
                "training_edges": train_graph.num_edges,
                "training_labels": len(history),
                "training_illicit": int(history["y"].sum()),
                "epochs_per_refit": epochs_per_refit,
                "final_training_loss": float(loss.item()),
                "refit_seconds": refit_seconds,
            })

            seed_metrics.append(metrics)
            seed_predictions.append(predictions)

        print(f"Seed {seed}: {policy} completed.")

    seed_metrics_df = pd.concat(seed_metrics, ignore_index=True)
    seed_predictions_df = pd.concat(
        seed_predictions, ignore_index=True
    )

    assert len(seed_metrics_df) == 45
    assert len(seed_predictions_df) == 16670 * 3
    assert not seed_predictions_df.duplicated(
        ["policy", "txId"]
    ).any()

    reference = test_reference.sort_values(
        "txId"
    ).reset_index(drop=True)

    for policy in ADAPTATION_POLICIES_D3:
        observed = (
            seed_predictions_df.loc[
                seed_predictions_df["policy"].eq(policy),
                ["txId", "time_step", "y"],
            ]
            .sort_values("txId")
            .reset_index(drop=True)
        )
        pd.testing.assert_frame_equal(reference, observed)

    seed_metrics_df.to_csv(
        output_dir / "metrics_by_step.csv", index=False
    )
    seed_predictions_df.to_csv(
        output_dir / "test_predictions.csv", index=False
    )
    pd.DataFrame(timing_rows).to_csv(
        output_dir / "refit_timing.csv", index=False
    )

    all_sage_adaptation_metrics.append(seed_metrics_df)

sage_d3_adaptation_metrics = pd.concat(
    all_sage_adaptation_metrics, ignore_index=True
)

assert len(sage_d3_adaptation_metrics) == 90
assert not sage_d3_adaptation_metrics.duplicated(
    ["model", "policy", "seed", "time_step"]
).any()

sage_d3_adaptation_metrics.to_csv(
    D3_RESULTS / "graphsage_adaptation_metrics_seeds1_2.csv",
    index=False,
)

sage_post_shift_summary = (
    sage_d3_adaptation_metrics[
        sage_d3_adaptation_metrics["time_step"].between(44, 49)
    ]
    .groupby(["seed", "policy"], as_index=False)
    .agg(
        mean_AP=("average_precision", "mean"),
        mean_F1=("f1", "mean"),
        mean_recall=("recall", "mean"),
    )
)

sage_post_shift_summary.to_csv(
    SAGE_D3_DIR / "adaptation_post_shift_summary.csv",
    index=False,
)

print("\nVerified GraphSAGE adaptation results:",
      len(sage_d3_adaptation_metrics))
print("Expected: 90")
display(sage_post_shift_summary.round(4))


Seed 1: 199 epochs per refit, threshold 0.90
Seed 1: expanding completed.
Seed 1: rolling completed.
Seed 1: recency completed.

Seed 2: 60 epochs per refit, threshold 0.85
Seed 2: expanding completed.
Seed 2: rolling completed.
Seed 2: recency completed.

Verified GraphSAGE adaptation results: 90
Expected: 90


,seed,policy,mean_AP,mean_F1,mean_recall
0,1,expanding,0.0858,0.0159,0.0099
1,1,recency,0.1607,0.0248,0.0146
2,1,rolling,0.1594,0.0056,0.0030
3,2,expanding,0.0765,0.0119,0.0099
4,2,recency,0.0806,0.0000,0.0000
5,2,rolling,0.0826,0.0000,0.0000


Audit and summarize all three seeds

In [11]:
import hashlib
import shutil

AUDIT_D3_DIR = D3_RESULTS / "evaluation"
AUDIT_D3_DIR.mkdir(exist_ok=True)

prefixes = {
    "XGBoost": "xgb",
    "MLP": "mlp",
    "GraphSAGE": "graphsage",
}

reference = (
    nodes.loc[
        nodes["time_step"].between(35, 49) & nodes["y"].ge(0),
        ["txId", "time_step", "y"],
    ]
    .sort_values("txId")
    .reset_index(drop=True)
)

all_audited_metrics = []
audit_records = []

# Preserve original seed-0 evidence alongside the new results.
baseline_copy = D3_RESULTS / "baseline_seed0"
baseline_copy.mkdir(exist_ok=True)

for model_name, prefix in prefixes.items():
    for seed in [0, 1, 2]:
        if seed == 0:
            frozen_path = (
                D2_RESULTS / f"{prefix}_frozen_test_predictions.csv"
            )
            adaptive_path = (
                D2_RESULTS / f"{prefix}_adaptation_seed0_predictions.csv"
            )
            frozen_metrics_path = (
                D2_RESULTS / f"{prefix}_frozen_metrics_by_step.csv"
            )
            adaptive_metrics_path = (
                D2_RESULTS / f"{prefix}_adaptation_seed0_metrics.csv"
            )
            settings_path = (
                D2_RESULTS / f"{prefix}_frozen_run_info.json"
            )

            for source in [
                frozen_path, adaptive_path,
                frozen_metrics_path, adaptive_metrics_path,
                settings_path,
            ]:
                shutil.copy2(source, baseline_copy / source.name)

        else:
            run_dir = (
                model_directories[model_name] / f"seed_{seed}"
            )
            frozen_path = run_dir / "frozen_test_predictions.csv"
            adaptive_path = (
                run_dir / "adaptation" / "test_predictions.csv"
            )
            frozen_metrics_path = (
                run_dir / "frozen_metrics_by_step.csv"
            )
            adaptive_metrics_path = (
                run_dir / "adaptation" / "metrics_by_step.csv"
            )
            settings_path = run_dir / "run_info.json"

        settings = json.loads(settings_path.read_text())
        threshold = float(settings["threshold"])

        frozen = pd.read_csv(frozen_path)
        frozen["policy"] = "frozen"

        adaptive = pd.read_csv(adaptive_path)
        assert set(adaptive["policy"]) == {
            "expanding", "rolling", "recency"
        }

        predictions = pd.concat(
            [frozen, adaptive], ignore_index=True
        )

        assert len(predictions) == 16670 * 4
        assert not predictions.duplicated(
            ["policy", "txId"]
        ).any()

        assert predictions["prediction"].isin([0, 1]).all()
        assert np.isfinite(predictions["score"]).all()

        np.testing.assert_array_equal(
            predictions["prediction"].to_numpy(),
            (
                predictions["score"].to_numpy() >= threshold
            ).astype(int),
        )

        saved_frozen = pd.read_csv(frozen_metrics_path)
        saved_frozen["policy"] = "frozen"
        saved_adaptive = pd.read_csv(adaptive_metrics_path)

        saved_metrics = pd.concat(
            [saved_frozen, saved_adaptive],
            ignore_index=True,
        )
        assert len(saved_metrics) == 60
        assert not saved_metrics.duplicated(
            ["policy", "time_step"]
        ).any()

        for policy in [
            "frozen", "expanding", "rolling", "recency"
        ]:
            subset = predictions.loc[
                predictions["policy"].eq(policy)
            ].copy()

            observed = (
                subset[["txId", "time_step", "y"]]
                .sort_values("txId")
                .reset_index(drop=True)
            )

            pd.testing.assert_frame_equal(
                reference, observed, check_dtype=False
            )

            recomputed = evaluate_d3_predictions(
                subset, model_name, policy, seed
            ).sort_values("time_step").reset_index(drop=True)

            saved = (
                saved_metrics.loc[
                    saved_metrics["policy"].eq(policy)
                ]
                .sort_values("time_step")
                .reset_index(drop=True)
            )

            np.testing.assert_array_equal(
                recomputed["time_step"], saved["time_step"]
            )

            for column in [
                "n_labeled", "n_illicit",
                "average_precision", "precision", "recall", "f1",
                "review_k",
                "precision_at_10pct", "recall_at_10pct",
            ]:
                np.testing.assert_allclose(
                    recomputed[column],
                    saved[column],
                    atol=1e-9,
                    rtol=1e-7,
                    equal_nan=True,
                    err_msg=(
                        f"{model_name}, seed {seed}, "
                        f"{policy}, {column}"
                    ),
                )

            all_audited_metrics.append(recomputed)

        audit_records.append({
            "model": model_name,
            "seed": seed,
            "prediction_rows_checked": len(predictions),
            "metric_rows_checked": 60,
            "threshold": threshold,
            "frozen_predictions_sha256": hashlib.sha256(
                frozen_path.read_bytes()
            ).hexdigest(),
            "adaptive_predictions_sha256": hashlib.sha256(
                adaptive_path.read_bytes()
            ).hexdigest(),
            "status": "passed",
        })

        print(f"Audit passed: {model_name}, seed {seed}")

all_seed_metrics = pd.concat(
    all_audited_metrics, ignore_index=True
)

assert len(all_seed_metrics) == 540
assert not all_seed_metrics.duplicated(
    ["model", "policy", "seed", "time_step"]
).any()

all_seed_metrics.to_csv(
    AUDIT_D3_DIR / "audited_metrics_all_seeds.csv",
    index=False,
)

audit_summary = {
    "status": "passed",
    "seeds": [0, 1, 2],
    "model_policy_seed_step_results": len(all_seed_metrics),
    "prediction_rows_checked": sum(
        row["prediction_rows_checked"] for row in audit_records
    ),
    "checks": audit_records,
}

assert audit_summary["prediction_rows_checked"] == 600120

(AUDIT_D3_DIR / "prediction_audit.json").write_text(
    json.dumps(audit_summary, indent=2)
)

# First average snapshots within each seed.
post_shift_per_seed = (
    all_seed_metrics[
        all_seed_metrics["time_step"].between(44, 49)
    ]
    .groupby(["model", "policy", "seed"], as_index=False)
    .agg(
        mean_AP=("average_precision", "mean"),
        mean_F1=("f1", "mean"),
        mean_recall=("recall", "mean"),
    )
)

# Then summarize variation across the three seeds.
three_seed_summary = (
    post_shift_per_seed
    .groupby(["model", "policy"], as_index=False)
    .agg(
        AP_mean=("mean_AP", "mean"),
        AP_sd=("mean_AP", "std"),
        F1_mean=("mean_F1", "mean"),
        F1_sd=("mean_F1", "std"),
        n_seeds=("seed", "nunique"),
    )
)

baseline_summary = (
    post_shift_per_seed[
        post_shift_per_seed["seed"].eq(0)
    ][["model", "policy", "mean_AP", "mean_F1"]]
    .rename(columns={
        "mean_AP": "D2_seed0_AP",
        "mean_F1": "D2_seed0_F1",
    })
)

comparison = baseline_summary.merge(
    three_seed_summary,
    on=["model", "policy"],
    validate="one_to_one",
)

assert len(comparison) == 12
assert comparison["n_seeds"].eq(3).all()

post_shift_per_seed.to_csv(
    AUDIT_D3_DIR / "post_shift_results_per_seed.csv",
    index=False,
)
comparison.to_csv(
    AUDIT_D3_DIR / "deliverable2_vs_deliverable3.csv",
    index=False,
)

print("\nAUDIT PASSED")
print("Verified model-policy-seed-step results:", len(all_seed_metrics))
print("Expected: 540")
print("Prediction rows checked:",
      audit_summary["prediction_rows_checked"])
print("\nPost-shift results: mean and sample SD across seeds")
display(comparison.round(4))

Audit passed: XGBoost, seed 0
Audit passed: XGBoost, seed 1
Audit passed: XGBoost, seed 2
Audit passed: MLP, seed 0
Audit passed: MLP, seed 1
Audit passed: MLP, seed 2
Audit passed: GraphSAGE, seed 0
Audit passed: GraphSAGE, seed 1
Audit passed: GraphSAGE, seed 2

AUDIT PASSED
Verified model-policy-seed-step results: 540
Expected: 540
Prediction rows checked: 600120

Post-shift results: mean and sample SD across seeds


,model,policy,D2_seed0_AP,D2_seed0_F1,AP_mean,AP_sd,F1_mean,F1_sd,n_seeds
0,GraphSAGE,expanding,0.0755,0.0000,0.0793,0.0057,0.0093,0.0083,3
1,GraphSAGE,frozen,0.0716,0.0129,0.0979,0.0482,0.0158,0.0025,3
2,GraphSAGE,recency,0.0817,0.0000,0.1077,0.0459,0.0083,0.0143,3
3,GraphSAGE,rolling,0.0881,0.0000,0.1100,0.0429,0.0019,0.0033,3
4,MLP,expanding,0.0684,0.0132,0.0656,0.0075,0.0122,0.0014,3
5,MLP,frozen,0.0938,0.0145,0.1078,0.0216,0.0167,0.0072,3
6,MLP,recency,0.1203,0.0234,0.1303,0.0355,0.0457,0.0298,3
7,MLP,rolling,0.1179,0.0205,0.1239,0.0341,0.0423,0.0214,3
8,XGBoost,expanding,0.1983,0.0872,0.1983,0.0000,0.0872,0.0000,3
9,XGBoost,frozen,0.0789,0.0242,0.0789,0.0000,0.0242,0.0000,3


In [12]:
from google.colab import files

archive_path = shutil.make_archive(
    str(PROJECT_DIR / "AML_GNN_Deliverable3_Results"),
    "zip",
    root_dir=str(D3_RESULTS),
)

print("Created:", archive_path)
files.download(archive_path)

Created: /content/drive/MyDrive/Colab Notebooks/AML_GNN_Project/AML_GNN_Deliverable3_Results.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>